# Notebook 6 — Context + Embedding RoBERTa

This notebook trains the first context-augmented model.

It uses the reusable mapping produced by the Wikimedia Context Bridge notebook:

```text
context_artifacts/context_mapped_examples.parquet
```

Model design:

```text
Retrieved Wikimedia context + comment
        ↓
      RoBERTa
        ↓
   CLS representation

subgroup_id
        ↓
subgroup embedding

[CLS ; subgroup_embedding]
        ↓
 classifier
        ↓
3-class probability distribution
```

This is the direct context version of the previous subgroup-embedding baseline.

The key question:

> Does retrieved subgroup-specific Wikimedia context improve distributional hate-speech prediction compared with the no-context embedding baseline?


## 1. Imports and Configuration

In [2]:
import ast
import json
import random
import itertools
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from scipy.spatial.distance import jensenshannon
from scipy.stats import entropy, pearsonr, spearmanr

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 220)

RANDOM_SEED = 42

MODEL_NAME = "roberta-base"
NUM_LABELS = 3

# Context inputs are longer. RTX 4060 8GB should usually handle this.
MAX_LENGTH = 384
BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 1

EPOCHS = 10
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
GRAD_CLIP = 1.0

SUBGROUP_DIM = 32
DROPOUT = 0.1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path("/home/shayan/Distributional-Hate-Speech-Prediction/data/processed")
CONTEXT_PATH = Path("/home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_artifacts/context_mapped_examples.parquet")

OUTPUT_DIR = Path("context_embedding_outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Device:", DEVICE)
print("Context file:", CONTEXT_PATH)
print("Output directory:", OUTPUT_DIR.resolve())


Device: cuda
Context file: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_artifacts/context_mapped_examples.parquet
Output directory: /home/shayan/Distributional-Hate-Speech-Prediction/notebooks/context_embedding_outputs


In [3]:
def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(RANDOM_SEED)


## 2. Load Context-Mapped Dataset

In [4]:
context_df = pd.read_parquet(CONTEXT_PATH)

print("Context dataset:", context_df.shape)
display(context_df.head(2))

required_columns = [
    "comment_id",
    "split",
    "subgroup",
    "text",
    "target_distribution",
    "target_majority_label",
    "context_input_text",
    "retrieved_article_titles",
    "retrieved_summaries",
]

missing = [col for col in required_columns if col not in context_df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Columns OK.")


Context dataset: (9090, 16)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text,tweet_token_length,context_input_token_length
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[1.0, 0.0, 0.0]",0,0.0,[Left-wing politics],[https://en.wikipedia.org/wiki/Left-wing_politics],[0.14819347858428955],"[Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a c...",### COMMENT TO CLASSIFY\n\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and ...,60,197
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[0.0, 0.0, 1.0]",2,2.0,[Left-wing politics],[https://en.wikipedia.org/wiki/Left-wing_politics],[0.14819347858428955],"[Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a c...",### COMMENT TO CLASSIFY\n\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and ...,60,195


Columns OK.


## 3. Sanity Checks on Context Mapping

In [5]:
print("Rows by split:")
display(context_df["split"].value_counts())

print("\nRows by subgroup:")
display(context_df["subgroup"].value_counts())

print("\nTarget majority distribution:")
display(context_df["target_majority_label"].value_counts(normalize=True).sort_index())

print("\nMissing context text:", context_df["context_input_text"].isna().sum())
print("Missing target distributions:", context_df["target_distribution"].isna().sum())


Rows by split:


split
train         6360
test          1384
validation    1346
Name: count, dtype: int64


Rows by subgroup:


subgroup
liberal                   2026
neutral                   1512
slightly_liberal          1477
extremely_liberal         1285
slightly_conservative     1079
conservative              1059
extremely_conservative     363
no_opinion                 289
Name: count, dtype: int64


Target majority distribution:


target_majority_label
0    0.734873
1    0.070957
2    0.194169
Name: proportion, dtype: float64


Missing context text: 0
Missing target distributions: 0


In [6]:
def safe_len(x):
    if isinstance(x, list):
        return len(x)
    if isinstance(x, np.ndarray):
        return len(x)
    if isinstance(x, str):
        try:
            return len(ast.literal_eval(x))
        except Exception:
            return np.nan
    return np.nan

print("Retrieved article count per row:")
display(context_df["retrieved_article_titles"].apply(safe_len).value_counts(dropna=False))

print("Retrieved summary count per row:")
display(context_df["retrieved_summaries"].apply(safe_len).value_counts(dropna=False))

print("\nExample full context input:")
print("=" * 100)
print(context_df.iloc[0]["context_input_text"])


Retrieved article count per row:


retrieved_article_titles
1    9090
Name: count, dtype: int64

Retrieved summary count per row:


retrieved_summaries
1    9090
Name: count, dtype: int64


Example full context input:
### COMMENT TO CLASSIFY
\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill dig him the hole to get there myself

### ANNOTATOR IDEOLOGY
extremely_liberal

### RETRIEVED BACKGROUND
Left-wing politics:
Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a concern for those disadvantaged by society and a belief in reducing or abolishing unjustified inequalities through radical means. Key concepts include justice, solidarity, cultural pluralism, and freedom from forceful control or coercion. The left is associated with popular or state control of major institutions, trade unions, and critiques of capitalism, globalization, and environmental d

## 4. Prepare Subgroup IDs and Splits

In [7]:
subgroup_to_id = {
    subgroup: idx
    for idx, subgroup in enumerate(sorted(context_df["subgroup"].unique()))
}

id_to_subgroup = {
    idx: subgroup
    for subgroup, idx in subgroup_to_id.items()
}

context_df["subgroup_id"] = context_df["subgroup"].map(subgroup_to_id)

print("Subgroup mapping:")
print(subgroup_to_id)

train_df = context_df[context_df["split"] == "train"].reset_index(drop=True)
val_df = context_df[context_df["split"].isin(["validation", "val", "dev"])].reset_index(drop=True)
test_df = context_df[context_df["split"] == "test"].reset_index(drop=True)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

assert len(train_df) > 0
assert len(val_df) > 0
assert len(test_df) > 0

display(pd.crosstab(context_df["split"], context_df["subgroup"]))


Subgroup mapping:
{'conservative': 0, 'extremely_conservative': 1, 'extremely_liberal': 2, 'liberal': 3, 'neutral': 4, 'no_opinion': 5, 'slightly_conservative': 6, 'slightly_liberal': 7}
Train: (6360, 17)
Val: (1346, 17)
Test: (1384, 17)


subgroup,conservative,extremely_conservative,extremely_liberal,liberal,neutral,no_opinion,slightly_conservative,slightly_liberal
split,,,,,,,,
test,163,46,197,308,257,46,154,213
train,739,252,907,1407,1027,206,756,1066
validation,157,65,181,311,228,37,169,198


## 5. Token Length Checks

In [8]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def count_tokens(text: str) -> int:
    return len(
        tokenizer(
            text,
            truncation=False,
            add_special_tokens=True,
        )["input_ids"]
    )

if "context_input_token_length" not in context_df.columns:
    context_df["context_input_token_length"] = context_df["context_input_text"].apply(count_tokens)

length_summary = {
    "mean": context_df["context_input_token_length"].mean(),
    "median": context_df["context_input_token_length"].median(),
    "p95": context_df["context_input_token_length"].quantile(0.95),
    "max": context_df["context_input_token_length"].max(),
    "pct_over_512": float((context_df["context_input_token_length"] > 512).mean()),
}

display(pd.DataFrame([length_summary]))

print("Rows over MAX_LENGTH:", (context_df["context_input_token_length"] > MAX_LENGTH).sum())


,mean,median,p95,max,pct_over_512
0,173.806051,168.0,225.0,328,0.0


Rows over MAX_LENGTH: 0


## 6. Dataset and Dataloader

In [9]:
def parse_distribution(value):
    if isinstance(value, np.ndarray):
        return value.astype(float).tolist()
    if isinstance(value, list):
        return [float(x) for x in value]
    if isinstance(value, str):
        parsed = ast.literal_eval(value)
        return [float(x) for x in parsed]
    raise TypeError(f"Unsupported distribution type: {type(value)}")


class ContextEmbeddingDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, tokenizer, max_length: int):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx: int):
        row = self.dataframe.iloc[idx]

        encoding = self.tokenizer(
            row["context_input_text"],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt",
        )

        target_distribution = parse_distribution(row["target_distribution"])

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "subgroup_id": torch.tensor(row["subgroup_id"], dtype=torch.long),
            "target_distribution": torch.tensor(target_distribution, dtype=torch.float),
        }


train_dataset = ContextEmbeddingDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = ContextEmbeddingDataset(val_df, tokenizer, MAX_LENGTH)
test_dataset = ContextEmbeddingDataset(test_df, tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

batch = next(iter(train_loader))
{k: v.shape for k, v in batch.items()}


{'input_ids': torch.Size([8, 384]),
 'attention_mask': torch.Size([8, 384]),
 'subgroup_id': torch.Size([8]),
 'target_distribution': torch.Size([8, 3])}

## 7. Context + Embedding Model

In [10]:
class ContextEmbeddingRoBERTa(nn.Module):
    def __init__(
        self,
        model_name: str,
        num_subgroups: int,
        subgroup_dim: int = 32,
        num_labels: int = 3,
        dropout: float = 0.1,
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.subgroup_embedding = nn.Embedding(
            num_embeddings=num_subgroups,
            embedding_dim=subgroup_dim,
        )

        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size + subgroup_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_labels),
        )

    def forward(self, input_ids, attention_mask, subgroup_id):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        subgroup_embedding = self.subgroup_embedding(subgroup_id)

        combined = torch.cat(
            [cls_embedding, subgroup_embedding],
            dim=-1,
        )

        logits = self.classifier(combined)

        return {
            "logits": logits,
            "log_probs": torch.log_softmax(logits, dim=-1),
            "probs": torch.softmax(logits, dim=-1),
        }


model = ContextEmbeddingRoBERTa(
    model_name=MODEL_NAME,
    num_subgroups=len(subgroup_to_id),
    subgroup_dim=SUBGROUP_DIM,
    num_labels=NUM_LABELS,
    dropout=DROPOUT,
).to(DEVICE)

criterion = nn.KLDivLoss(reduction="batchmean")

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

num_update_steps_per_epoch = int(np.ceil(len(train_loader) / GRADIENT_ACCUMULATION_STEPS))
num_training_steps = num_update_steps_per_epoch * EPOCHS
num_warmup_steps = int(WARMUP_RATIO * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps,
)

print("Train batches per epoch:", len(train_loader))
print("Gradient accumulation:", GRADIENT_ACCUMULATION_STEPS)
print("Optimizer update steps per epoch:", num_update_steps_per_epoch)
print("Training steps:", num_training_steps)
print("Warmup steps:", num_warmup_steps)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Train batches per epoch: 795
Gradient accumulation: 1
Optimizer update steps per epoch: 795
Training steps: 7950
Warmup steps: 795


## 8. Metrics

In [11]:
EPS = 1e-12


def kl_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_true = np.clip(y_true, EPS, 1.0)
    y_pred = np.clip(y_pred, EPS, 1.0)
    return np.sum(y_true * np.log(y_true / y_pred), axis=1)


def js_divergence(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    values = []
    for true_dist, pred_dist in zip(y_true, y_pred):
        values.append(jensenshannon(true_dist, pred_dist, base=2) ** 2)
    return np.array(values)


def cross_entropy(y_true: np.ndarray, y_pred: np.ndarray) -> np.ndarray:
    y_pred = np.clip(y_pred, EPS, 1.0)
    return -np.sum(y_true * np.log(y_pred), axis=1)


def expected_scores(distributions: np.ndarray) -> np.ndarray:
    labels = np.arange(distributions.shape[1])
    return distributions @ labels


def entropy_values(distributions: np.ndarray) -> np.ndarray:
    return np.array([entropy(dist, base=2) for dist in distributions])


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    true_labels = np.argmax(y_true, axis=1)
    pred_labels = np.argmax(y_pred, axis=1)

    true_expected = expected_scores(y_true)
    pred_expected = expected_scores(y_pred)

    true_entropy = entropy_values(y_true)
    pred_entropy = entropy_values(y_pred)

    metrics = {
        "kl_mean": float(kl_divergence(y_true, y_pred).mean()),
        "js_mean": float(js_divergence(y_true, y_pred).mean()),
        "cross_entropy_mean": float(cross_entropy(y_true, y_pred).mean()),
        "accuracy": float(accuracy_score(true_labels, pred_labels)),
        "macro_f1": float(f1_score(true_labels, pred_labels, average="macro", zero_division=0)),
        "expected_label_mae": float(mean_absolute_error(true_expected, pred_expected)),
    }

    if len(np.unique(true_entropy)) > 1 and len(np.unique(pred_entropy)) > 1:
        metrics["entropy_pearson"] = float(pearsonr(true_entropy, pred_entropy).statistic)
        metrics["entropy_spearman"] = float(spearmanr(true_entropy, pred_entropy).statistic)
    else:
        metrics["entropy_pearson"] = np.nan
        metrics["entropy_spearman"] = np.nan

    return metrics


## 9. Training Helpers

In [12]:
def run_epoch(model, dataloader, optimizer=None, scheduler=None):
    is_training = optimizer is not None

    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_targets = []
    all_predictions = []

    if is_training:
        optimizer.zero_grad()

    for step, batch in enumerate(dataloader):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        subgroup_id = batch["subgroup_id"].to(DEVICE)
        targets = batch["target_distribution"].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                subgroup_id=subgroup_id,
            )

            raw_loss = criterion(outputs["log_probs"], targets)

            if is_training:
                loss = raw_loss / GRADIENT_ACCUMULATION_STEPS
                loss.backward()

                should_step = (
                    ((step + 1) % GRADIENT_ACCUMULATION_STEPS == 0)
                    or ((step + 1) == len(dataloader))
                )

                if should_step:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    optimizer.step()
                    optimizer.zero_grad()

                    if scheduler is not None:
                        scheduler.step()

        total_loss += raw_loss.item() * input_ids.size(0)
        all_targets.append(targets.detach().cpu().numpy())
        all_predictions.append(outputs["probs"].detach().cpu().numpy())

    avg_loss = total_loss / len(dataloader.dataset)

    y_true = np.vstack(all_targets)
    y_pred = np.vstack(all_predictions)

    metrics = compute_metrics(y_true, y_pred)
    metrics["loss"] = float(avg_loss)

    return metrics, y_true, y_pred


## 10. Train Model

In [13]:
best_val_kl = float("inf")
best_model_path = OUTPUT_DIR / "context_embedding_best_model.pt"

history = []

for epoch in range(1, EPOCHS + 1):

    train_metrics, _, _ = run_epoch(
        model,
        train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
    )

    val_metrics, _, _ = run_epoch(
        model,
        val_loader,
        optimizer=None,
        scheduler=None,
    )

    row = {
        "epoch": epoch,
        **{f"train_{k}": v for k, v in train_metrics.items()},
        **{f"val_{k}": v for k, v in val_metrics.items()},
    }

    history.append(row)

    print("=" * 80)
    print(f"Epoch {epoch}/{EPOCHS}")
    print(f"Train KL: {train_metrics['kl_mean']:.4f} | Val KL: {val_metrics['kl_mean']:.4f}")
    print(f"Train JS: {train_metrics['js_mean']:.4f} | Val JS: {val_metrics['js_mean']:.4f}")
    print(f"Val Acc: {val_metrics['accuracy']:.4f} | Val Macro F1: {val_metrics['macro_f1']:.4f}")

    if val_metrics["kl_mean"] < best_val_kl:
        best_val_kl = val_metrics["kl_mean"]
        torch.save(model.state_dict(), best_model_path)
        print(f"Saved new best model to {best_model_path}")

history_df = pd.DataFrame(history)
display(history_df)

history_path = OUTPUT_DIR / "context_embedding_training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved:", history_path)


Epoch 1/10
Train KL: 0.7284 | Val KL: 0.6278
Train JS: 0.2834 | Val JS: 0.2118
Val Acc: 0.7727 | Val Macro F1: 0.4546
Saved new best model to context_embedding_outputs/context_embedding_best_model.pt
Epoch 2/10
Train KL: 0.6121 | Val KL: 0.6599
Train JS: 0.2236 | Val JS: 0.2155
Val Acc: 0.7489 | Val Macro F1: 0.3176
Epoch 3/10
Train KL: 0.5550 | Val KL: 0.6502
Train JS: 0.1992 | Val JS: 0.2058
Val Acc: 0.7756 | Val Macro F1: 0.4683
Epoch 4/10
Train KL: 0.5179 | Val KL: 0.6310
Train JS: 0.1827 | Val JS: 0.2239
Val Acc: 0.7519 | Val Macro F1: 0.4646
Epoch 5/10
Train KL: 0.4826 | Val KL: 0.6675
Train JS: 0.1723 | Val JS: 0.2070
Val Acc: 0.7808 | Val Macro F1: 0.4745
Epoch 6/10
Train KL: 0.4562 | Val KL: 0.6325
Train JS: 0.1606 | Val JS: 0.2084
Val Acc: 0.7823 | Val Macro F1: 0.4528


KeyboardInterrupt: 

## 11. Test Evaluation

In [14]:
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))

test_metrics, y_true_test, y_pred_test = run_epoch(
    model,
    test_loader,
    optimizer=None,
    scheduler=None,
)

display(test_metrics)

metrics_path = OUTPUT_DIR / "context_embedding_test_metrics.json"

with open(metrics_path, "w") as f:
    json.dump(test_metrics, f, indent=2)

print("Saved:", metrics_path)


/tmp/ipykernel_82902/3133918950.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))


{'kl_mean': 0.6376786828041077,
 'js_mean': 0.2208747622047546,
 'cross_entropy_mean': 0.6621034741401672,
 'accuracy': 0.7644508670520231,
 'macro_f1': 0.43716456772410855,
 'expected_label_mae': 0.4872280428235855,
 'entropy_pearson': 0.06039579073056483,
 'entropy_spearman': 0.06577192911669683,
 'loss': 0.6376786724249751}

Saved: context_embedding_outputs/context_embedding_test_metrics.json


## 12. Save Predictions

In [15]:
predictions_df = test_df.copy()

predictions_df["true_distribution"] = list(y_true_test)
predictions_df["pred_distribution"] = list(y_pred_test)

predictions_df["pred_majority_label"] = np.argmax(y_pred_test, axis=1)
predictions_df["pred_expected_label"] = expected_scores(y_pred_test)
predictions_df["pred_entropy"] = entropy_values(y_pred_test)

predictions_df["kl"] = kl_divergence(y_true_test, y_pred_test)
predictions_df["js"] = js_divergence(y_true_test, y_pred_test)
predictions_df["cross_entropy"] = cross_entropy(y_true_test, y_pred_test)

display(predictions_df.head())

predictions_path = OUTPUT_DIR / "context_embedding_test_predictions.parquet"
predictions_df.to_parquet(predictions_path, index=False)

csv_predictions_path = OUTPUT_DIR / "context_embedding_test_predictions.csv"
predictions_df.to_csv(csv_predictions_path, index=False)

print("Saved:", predictions_path)
print("Saved:", csv_predictions_path)


,experiment,comment_id,split,subgroup,subgroup_count,text,target_distribution,target_majority_label,target_expected_label,retrieved_article_titles,retrieved_page_urls,retrieved_similarities,retrieved_summaries,context_input_text,tweet_token_length,context_input_token_length,subgroup_id,true_distribution,pred_distribution,pred_majority_label,pred_expected_label,pred_entropy,kl,js,cross_entropy
0,immigration,7,test,extremely_liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[1.0, 0.0, 0.0]",0,0.0,[Left-wing politics],[https://en.wikipedia.org/wiki/Left-wing_politics],[0.14819347858428955],"[Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a c...",### COMMENT TO CLASSIFY\n\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and ...,60,197,2,"[1.0, 0.0, 0.0]","[0.29029524, 0.07060897, 0.6390958]",2,1.348801,1.200803,1.236857,0.503776,1.236857
1,immigration,7,test,liberal,1,\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and go straight to hell. ill ...,"[0.0, 0.0, 1.0]",2,2.0,[Left-wing politics],[https://en.wikipedia.org/wiki/Left-wing_politics],[0.14819347858428955],"[Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a c...",### COMMENT TO CLASSIFY\n\*points to posters asking for palestinian rights and freedom\* theyre antisemetic and want to kill all the jews and dont acknowledge our right to exist ​ cant wait for this prick to die and ...,60,195,3,"[0.0, 0.0, 1.0]","[0.29658657, 0.079431534, 0.6239819]",2,1.327395,1.234884,0.471634,0.219697,0.471634
2,immigration,66,test,extremely_liberal,1,Get ready to be flogged. India is ready to flog all cow dung eating shameless fucktards,"[0.0, 0.0, 1.0]",2,2.0,[Free migration],[https://en.wikipedia.org/wiki/Free_migration],[0.08663808554410934],"[The concept of free migration has been debated globally, with various perspectives on its moral, economic, and cultural implications. Some argue that it's a human right, citing religious teachings such as Buddhism a...",### COMMENT TO CLASSIFY\nGet ready to be flogged. India is ready to flog all cow dung eating shameless fucktards\n\n### ANNOTATOR IDEOLOGY\nextremely_liberal\n\n### RETRIEVED BACKGROUND\nFree migration:\nThe concept ...,24,157,2,"[0.0, 0.0, 1.0]","[0.20812744, 0.0791222, 0.7127504]",2,1.504623,1.109067,0.338624,0.161083,0.338624
3,immigration,66,test,slightly_liberal,1,Get ready to be flogged. India is ready to flog all cow dung eating shameless fucktards,"[1.0, 0.0, 0.0]",0,0.0,[Left-wing politics],[https://en.wikipedia.org/wiki/Left-wing_politics],[0.07852911949157715],"[Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a c...",### COMMENT TO CLASSIFY\nGet ready to be flogged. India is ready to flog all cow dung eating shameless fucktards\n\n### ANNOTATOR IDEOLOGY\nslightly_liberal\n\n### RETRIEVED BACKGROUND\nLeft-wing politics:\nLeft-wing...,24,162,7,"[1.0, 0.0, 0.0]","[0.23211709, 0.084755525, 0.6831273]",2,1.451010,1.166430,1.460513,0.569936,1.460513
4,immigration,108,test,conservative,1,"I am secretly hoping he is simply saying that to deter them from coming, he keeps suspending the Haiti deportatio

Saved: context_embedding_outputs/context_embedding_test_predictions.parquet
Saved: context_embedding_outputs/context_embedding_test_predictions.csv


## 13. Standard Diagnostics

In [16]:
true_labels = np.argmax(y_true_test, axis=1)
pred_labels = np.argmax(y_pred_test, axis=1)

print("Confusion matrix:")
print(confusion_matrix(true_labels, pred_labels, labels=[0, 1, 2]))

print("\nClassification report:")
print(classification_report(true_labels, pred_labels, labels=[0, 1, 2], zero_division=0))

print("\nPredicted label distribution:")
display(pd.Series(pred_labels).value_counts(normalize=True).sort_index())

print("\nTrue label distribution:")
display(pd.Series(true_labels).value_counts(normalize=True).sort_index())


Confusion matrix:
[[955   0  74]
 [ 73   0  22]
 [157   0 103]]

Classification report:
              precision    recall  f1-score   support

           0       0.81      0.93      0.86      1029
           1       0.00      0.00      0.00        95
           2       0.52      0.40      0.45       260

    accuracy                           0.76      1384
   macro avg       0.44      0.44      0.44      1384
weighted avg       0.70      0.76      0.73      1384


Predicted label distribution:


0    0.856214
2    0.143786
Name: proportion, dtype: float64


True label distribution:


0    0.743497
1    0.068642
2    0.187861
Name: proportion, dtype: float64

In [17]:
print("Mean KL by true majority label:")
display(
    predictions_df
    .groupby("target_majority_label")
    .agg(
        n=("comment_id", "count"),
        mean_kl=("kl", "mean"),
        mean_js=("js", "mean"),
        mean_target_entropy=("target_distribution", lambda x: np.mean([entropy(parse_distribution(v), base=2) for v in x])),
        mean_pred_entropy=("pred_entropy", "mean"),
    )
)

print("Average predicted distribution by true majority label:")
for label in [0, 1, 2]:
    subset = predictions_df[predictions_df["target_majority_label"] == label]
    avg_pred = np.vstack(subset["pred_distribution"].to_numpy()).mean(axis=0)
    print(label, avg_pred)


Mean KL by true majority label:


,n,mean_kl,mean_js,mean_target_entropy,mean_pred_entropy
target_majority_label,,,,,
0,1029,0.223301,0.096867,0.035754,0.625278
1,95,3.135908,0.857692,0.052632,0.801821
2,260,1.364843,0.478976,0.026836,0.990089


Average predicted distribution by true majority label:
0 [0.8394679  0.03526596 0.1252664 ]
1 [0.7200461  0.04374392 0.23621014]
2 [0.58946174 0.05279154 0.35774708]


In [18]:
print("Performance by subgroup:")

subgroup_rows = []

for subgroup, group in predictions_df.groupby("subgroup"):
    y_true = np.vstack(group["true_distribution"].to_numpy())
    y_pred = np.vstack(group["pred_distribution"].to_numpy())

    row = {
        "subgroup": subgroup,
        "n": len(group),
        **compute_metrics(y_true, y_pred),
    }

    subgroup_rows.append(row)

subgroup_metrics_df = pd.DataFrame(subgroup_rows).sort_values("kl_mean")
display(subgroup_metrics_df)

subgroup_metrics_path = OUTPUT_DIR / "context_embedding_subgroup_metrics.csv"
subgroup_metrics_df.to_csv(subgroup_metrics_path, index=False)
print("Saved:", subgroup_metrics_path)


Performance by subgroup:


,subgroup,n,kl_mean,js_mean,cross_entropy_mean,accuracy,macro_f1,expected_label_mae,entropy_pearson,entropy_spearman
1,extremely_conservative,46,0.401958,0.159982,0.412504,0.869565,0.555473,0.402010,0.141894,0.129130
0,conservative,163,0.501368,0.178081,0.513300,0.822086,0.492813,0.387573,0.007675,0.023560
6,slightly_conservative,154,0.558197,0.199902,0.569756,0.785714,0.458165,0.455164,-0.043501,-0.065983
5,no_opinion,46,0.588833,0.204117,0.607752,0.782609,0.476265,0.448306,-0.051986,-0.086384
3,liberal,308,0.628563,0.210665,0.668825,0.785714,0.439938,0.454997,0.098248,0.120504
4,neutral,257,0.690673,0.240160,0.724095,0.739300,0.424593,0.532163,0.100493,0.096711
7,slightly_liberal,213,0.698896,0.243204,0.716042,0.718310,0.399490,0.535294,0.108694,0.075005
2,extremely_liberal,197,0.757972,0.257471,0.778689,0.720812,0.385129,0.563537,-0.011127,0.016816


Saved: context_embedding_outputs/context_embedding_subgroup_metrics.csv


## 14. Counterfactual Subgroup Sensitivity

In [19]:
@torch.no_grad()
def predict_distribution(context_text: str, subgroup: str) -> np.ndarray:
    model.eval()

    encoding = tokenizer(
        context_text,
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    input_ids = encoding["input_ids"].to(DEVICE)
    attention_mask = encoding["attention_mask"].to(DEVICE)

    subgroup_id = torch.tensor(
        [subgroup_to_id[subgroup]],
        dtype=torch.long,
    ).to(DEVICE)

    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        subgroup_id=subgroup_id,
    )

    return outputs["probs"].detach().cpu().numpy()[0]


### Important note on counterfactuals with context

For this context model, counterfactual subgroup sensitivity should change both:

1. the subgroup embedding, and  
2. the retrieved context.

Therefore, when comparing groups, we use the context text mapped for that `(comment_id, subgroup)` row.


In [20]:
subgroups = list(subgroup_to_id.keys())

# Lookup context text by comment_id and subgroup.
context_lookup = {
    (row["comment_id"], row["subgroup"]): row["context_input_text"]
    for _, row in context_df.iterrows()
}

unique_test_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

counterfactual_rows = []

for _, row in unique_test_comments.iterrows():
    comment_id = row["comment_id"]

    available_subgroups = [
        subgroup for subgroup in subgroups
        if (comment_id, subgroup) in context_lookup
    ]

    if len(available_subgroups) < 2:
        continue

    preds_by_group = {}

    for subgroup in available_subgroups:
        context_text = context_lookup[(comment_id, subgroup)]
        preds_by_group[subgroup] = predict_distribution(context_text, subgroup)

    pairwise_js = []

    for group_a, group_b in itertools.combinations(available_subgroups, 2):
        js = jensenshannon(
            preds_by_group[group_a],
            preds_by_group[group_b],
            base=2,
        ) ** 2

        pairwise_js.append(js)

    counterfactual_rows.append({
        "comment_id": comment_id,
        "text": row["text"],
        "n_available_subgroups": len(available_subgroups),
        "mean_pairwise_js": float(np.mean(pairwise_js)),
        "max_pairwise_js": float(np.max(pairwise_js)),
    })

cf_df = pd.DataFrame(counterfactual_rows)

print("Counterfactual rows:", cf_df.shape)
print("Mean counterfactual JS:", cf_df["mean_pairwise_js"].mean())
print("Median counterfactual JS:", cf_df["mean_pairwise_js"].median())
print("Mean max-pairwise JS:", cf_df["max_pairwise_js"].mean())
print("Median max-pairwise JS:", cf_df["max_pairwise_js"].median())

display(
    cf_df
    .sort_values("max_pairwise_js", ascending=False)
    .head(30)
)

cf_path = OUTPUT_DIR / "context_embedding_counterfactual_js.csv"
cf_df.to_csv(cf_path, index=False)
print("Saved:", cf_path)


Counterfactual rows: (590, 5)
Mean counterfactual JS: 0.0017183567690534781
Median counterfactual JS: 0.00026826417359894993
Mean max-pairwise JS: 0.0020757969296184603
Median max-pairwise JS: 0.00031629624915360306


,comment_id,text,n_available_subgroups,mean_pairwise_js,max_pairwise_js
403,42177,jeezus those pahang players can't score normal goals,3,0.021711,0.041967
272,32694,Fuck Israel. URL,4,0.018891,0.038681
208,25480,Move out of this country then tough guy!! You are a piece of shit!,2,0.032432,0.032432
293,34709,Somalia must be a giant shithole why do we take in this trash,3,0.014275,0.024308
61,8223,"It's funny because the UK has allowed foreign powers to flood their once great country with inbred, low-IQ savages.",4,0.011045,0.024258
483,45185,Shoot a few of those border jumpers and the word will get out. Fuck them. Remember the Alamo!!!,2,0.023616,0.023616
530,47662,more attacks on Americans by the traitor in chief all to the glory of stupid white trash,2,0.023508,0.023508
294,35005,well you know TOMI not all of us had the benefit of our ancestors eradicating native peoples so they could settle their genocidal asses upon their lands.,2,0.021849,0.021849
56,7741,Now put this worthless immigrant trash to death.,3,0.014200,0.021560
163,20012,""": No need to thank me, killing niggers is it's own reward."" Dis my pal from our graduation when we was 26",3,0.011214,0.020602


Saved: context_embedding_outputs/context_embedding_counterfactual_js.csv


## 15. EL vs EC Counterfactual Analysis

In [21]:
def pairwise_counterfactual_js(group_a: str, group_b: str) -> pd.DataFrame:
    rows = []

    unique_comments = test_df[["comment_id", "text"]].drop_duplicates("comment_id").reset_index(drop=True)

    for _, row in unique_comments.iterrows():
        comment_id = row["comment_id"]

        if (comment_id, group_a) not in context_lookup:
            continue

        if (comment_id, group_b) not in context_lookup:
            continue

        context_a = context_lookup[(comment_id, group_a)]
        context_b = context_lookup[(comment_id, group_b)]

        pred_a = predict_distribution(context_a, group_a)
        pred_b = predict_distribution(context_b, group_b)

        js = jensenshannon(
            pred_a,
            pred_b,
            base=2,
        ) ** 2

        rows.append({
            "comment_id": comment_id,
            "text": row["text"],
            "group_a": group_a,
            "group_b": group_b,
            "js": float(js),
            f"pred_{group_a}": pred_a,
            f"pred_{group_b}": pred_b,
            f"context_{group_a}": context_a,
            f"context_{group_b}": context_b,
        })

    return pd.DataFrame(rows)


print("Available subgroups:")
print(subgroup_to_id)


Available subgroups:
{'conservative': 0, 'extremely_conservative': 1, 'extremely_liberal': 2, 'liberal': 3, 'neutral': 4, 'no_opinion': 5, 'slightly_conservative': 6, 'slightly_liberal': 7}


In [22]:
if "extremely_liberal" in subgroup_to_id and "extremely_conservative" in subgroup_to_id:
    extreme_df = pairwise_counterfactual_js(
        "extremely_liberal",
        "extremely_conservative",
    )

    print("EL vs EC rows:", extreme_df.shape)
    print("Extreme liberal vs extreme conservative mean JS:", extreme_df["js"].mean())
    print("Extreme liberal vs extreme conservative median JS:", extreme_df["js"].median())

    display(
        extreme_df
        .sort_values("js", ascending=False)
        .head(30)
    )

    extreme_path = OUTPUT_DIR / "context_embedding_EL_vs_EC_counterfactual.csv"
    extreme_df.to_csv(extreme_path, index=False)
    print("Saved:", extreme_path)


EL vs EC rows: (8, 9)
Extreme liberal vs extreme conservative mean JS: 0.0011011997489564836
Extreme liberal vs extreme conservative median JS: 0.0009526038414423379


,comment_id,text,group_a,group_b,js,pred_extremely_liberal,pred_extremely_conservative,context_extremely_liberal,context_extremely_conservative
7,49332,Send all these asses back !,extremely_liberal,extremely_conservative,0.002679,"[0.36457667, 0.06890402, 0.56651926]","[0.3074006, 0.07150222, 0.62109715]","### COMMENT TO CLASSIFY\nSend all these asses back !\n\n### ANNOTATOR IDEOLOGY\nextremely_liberal\n\n### RETRIEVED BACKGROUND\nFree migration:\nThe concept of free migration has been debated globally, with various pe...",### COMMENT TO CLASSIFY\nSend all these asses back !\n\n### ANNOTATOR IDEOLOGY\nextremely_conservative\n\n### RETRIEVED BACKGROUND\nOpposition to immigration:\nNational identity plays a significant role in shaping at...
4,37246,racists b like go back to ur country like ok stop bombing it n maybe i will,extremely_liberal,extremely_conservative,0.002008,"[0.7363946, 0.047278102, 0.21632731]","[0.69144535, 0.04775684, 0.2607978]",### COMMENT TO CLASSIFY\nracists b like go back to ur country like ok stop bombing it n maybe i will\n\n### ANNOTATOR IDEOLOGY\nextremely_liberal\n\n### RETRIEVED BACKGROUND\nFree migration:\nThe concept of free migr...,### COMMENT TO CLASSIFY\nracists b like go back to ur country like ok stop bombing it n maybe i will\n\n### ANNOTATOR IDEOLOGY\nextremely_conservative\n\n### RETRIEVED BACKGROUND\nOpposition to immigration:\nNational...
0,13695,Yes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithole as their original count...,extremely_liberal,extremely_conservative,0.001475,"[0.82507026, 0.038044184, 0.13688551]","[0.7930761, 0.037540104, 0.16938376]",### COMMENT TO CLASSIFY\nYes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithol...,### COMMENT TO CLASSIFY\nYes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithol...
6,43958,"Donald trump is the man , he's right if your not happy in the country you are welcomed to fuck of back to where you came from it's not racist or anything of the sort just clear and simple don't matter who you are fuc...",extremely_liberal,extremely_conservative,0.001120,"[0.7210448, 0.046371896, 0.23258336]","[0.6872679, 0.04631285, 0.2664192]","### COMMENT TO CLASSIFY\nDonald trump is the man , he's right if your not happy in the country you are welcomed to fuck of back to where you came from it's not racist or anything of the sort just clear and simple don...","### COMMENT TO CLASSIFY\nDonald trump is the man , he's right if your not happy in the country you are welcomed to fuck of back to where you came from it's not racist or anything of the sort just clear and simple don..."
2,20033,"STOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE",extremely_liberal,extremely_conservative,0.000785,"[0.3831544, 0.063040204, 0.5538054]","[0.3527059, 0.061515607, 0.58577853]","### COMMENT TO CLASSIFY\nSTOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE\n\n### ANNOTATOR IDEOLOGY\nextremely_liberal\n\n### RETRIEVED BACKGROUND\nLeft-wing politics:\nLeft-wing politics emphasizes social equal...","### COMMENT TO CLASSIFY\nSTOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE\n\n### ANNOTATOR IDEOLOGY\nextremely_conservative\n\n### RETRIEVED BACKGROUND\nNational conservatism:\nNational conservatism prioritizes ..."
1,20016,Keep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of these fucking parasites ou...,extremely_liberal,extremely_conservative,0.000368,"[0.21677317, 0.07754286, 0.705684]","[0.19847782, 0.07862479, 0.72289735]",### COMMENT TO

Saved: context_embedding_outputs/context_embedding_EL_vs_EC_counterfactual.csv


## 16. True EL/EC Disagreement vs Model Counterfactual Disagreement

In [23]:
def valid_dist(x):
    if x is None:
        return False
    try:
        arr = np.array(parse_distribution(x), dtype=float)
    except Exception:
        return False
    if arr.shape[0] != NUM_LABELS:
        return False
    if np.isnan(arr).any():
        return False
    if arr.sum() <= 0:
        return False
    return True


def true_pairwise_disagreement_from_context_df(
    full_df: pd.DataFrame,
    group_a: str,
    group_b: str,
) -> pd.DataFrame:
    rows = []

    grouped = full_df.groupby("comment_id")

    for comment_id, group in grouped:
        if group_a not in set(group["subgroup"]):
            continue
        if group_b not in set(group["subgroup"]):
            continue

        row_a = group[group["subgroup"] == group_a].iloc[0]
        row_b = group[group["subgroup"] == group_b].iloc[0]

        dist_a = parse_distribution(row_a["target_distribution"])
        dist_b = parse_distribution(row_b["target_distribution"])

        if not valid_dist(dist_a) or not valid_dist(dist_b):
            continue

        dist_a = np.array(dist_a, dtype=float)
        dist_b = np.array(dist_b, dtype=float)

        true_js = jensenshannon(dist_a, dist_b, base=2) ** 2

        rows.append({
            "comment_id": comment_id,
            "text": row_a["text"],
            "true_js": float(true_js),
            f"{group_a}_true_dist": dist_a,
            f"{group_b}_true_dist": dist_b,
            f"{group_a}_true_label": int(np.argmax(dist_a)),
            f"{group_b}_true_label": int(np.argmax(dist_b)),
        })

    return pd.DataFrame(rows)


In [24]:
if "extremely_liberal" in subgroup_to_id and "extremely_conservative" in subgroup_to_id:
    true_df = true_pairwise_disagreement_from_context_df(
        context_df,
        "extremely_liberal",
        "extremely_conservative",
    )

    comparison_df = true_df.merge(
        extreme_df[[
            "comment_id",
            "js",
            "pred_extremely_liberal",
            "pred_extremely_conservative",
        ]],
        on="comment_id",
        how="inner",
    )

    comparison_df = comparison_df.rename(columns={"js": "model_js"})

    nonzero = comparison_df[comparison_df["true_js"] > 0].copy()
    nonzero["capture_ratio"] = nonzero["model_js"] / nonzero["true_js"]

    print("N true EL/EC overlap:", len(true_df))
    print("N comparison:", len(comparison_df))
    print("Mean true JS:", comparison_df["true_js"].mean())
    print("Mean model JS:", comparison_df["model_js"].mean())

    if len(nonzero) > 0:
        print("Mean capture ratio, true_js > 0:", nonzero["capture_ratio"].mean())
        print("Median capture ratio, true_js > 0:", nonzero["capture_ratio"].median())

    comparison_df["label_pair"] = (
        comparison_df["extremely_liberal_true_label"].astype(str)
        + "-"
        + comparison_df["extremely_conservative_true_label"].astype(str)
    )

    display(
        comparison_df
        .groupby("label_pair")
        .agg(
            n=("comment_id", "count"),
            mean_true_js=("true_js", "mean"),
            mean_model_js=("model_js", "mean"),
        )
        .sort_values("mean_true_js", ascending=False)
    )

    display(
        comparison_df
        .sort_values("true_js", ascending=False)
        .head(30)
    )

    comparison_path = OUTPUT_DIR / "context_embedding_true_vs_model_EL_EC.csv"
    comparison_df.to_csv(comparison_path, index=False)
    print("Saved:", comparison_path)


N true EL/EC overlap: 101
N comparison: 8
Mean true JS: 0.16448066401642414
Mean model JS: 0.0011011997489564836
Mean capture ratio, true_js > 0: 0.029355164232818216
Median capture ratio, true_js > 0: 0.004740057187486024


,n,mean_true_js,mean_model_js
label_pair,,,
0-1,1,1.000000,0.002679
0-0,4,0.077820,0.001184
2-2,3,0.001522,0.000465


,comment_id,text,true_js,extremely_liberal_true_dist,extremely_conservative_true_dist,extremely_liberal_true_label,extremely_conservative_true_label,model_js,pred_extremely_liberal,pred_extremely_conservative,label_pair
7,49332,Send all these asses back !,1.000000,"[1.0, 0.0, 0.0]","[0.0, 1.0, 0.0]",0,1,0.002679,"[0.36457667, 0.06890402, 0.56651926]","[0.3074006, 0.07150222, 0.62109715]",0-1
0,13695,Yes so every lazy poverty stricken uneducated third world citizen in the entire world from Africa to South America can come here and get welfare and turn America into just as bad of a shithole as their original count...,0.311278,"[0.5, 0.0, 0.5]","[1.0, 0.0, 0.0]",0,0,0.001475,"[0.82507026, 0.038044184, 0.13688551]","[0.7930761, 0.037540104, 0.16938376]",0-0
1,20016,Keep the fucking Mexican scum rapists and murderers out of the country and the fucking Arab Islamic terrorists out of the U.S. too!! No Syrian refugees either!! Build a wall and keep all of these fucking parasites ou...,0.004567,"[0.08421052992343903, 0.10526315867900848, 0.8105263113975525]","[0.06666667014360428, 0.06666667014360428, 0.8666666746139526]",2,2,0.000368,"[0.21677317, 0.07754286, 0.705684]","[0.19847782, 0.07862479, 0.72289735]",2-2
2,20033,"STOP SUCKING THAT FATASS' DICK, YOU NIGGERLOVING KIKE",0.000000,"[0.0, 0.0, 1.0]","[0.0, 0.0, 1.0]",2,2,0.000785,"[0.3831544, 0.063040204, 0.5538054]","[0.3527059, 0.061515607, 0.58577853]",2-2
3,20062,I've always been proud of the Igbo tribe and our culture. Yesterday made me even more proud. You people are too fine abeg. OgbakoUmuIgboDay,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0.000130,"[0.9263814, 0.030398967, 0.04321958]","[0.9246594, 0.027600069, 0.047740553]",0-0
4,37246,racists b like go back to ur country like ok stop bombing it n maybe i will,0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0.002008,"[0.7363946, 0.047278102, 0.21632731]","[0.69144535, 0.04775684, 0.2607978]",0-0
5,38441,Indian cunts and dickheads orgasming over developing the occupied and annexed Kashmir so they can settle there while the Kashmiri natives remain locked down in their houses with the Indian occupation forces roaming t...,0.000000,"[0.0, 0.0, 1.0]","[0.0, 0.0, 1.0]",2,2,0.000243,"[0.28451982, 0.07453978, 0.64094037]","[0.26871943, 0.07360146, 0.6576791]",2-2
6,43958,"Donald trump is the man , he's right if your not happy in the country you are welcomed to fuck of back to where you came from it's not racist or anything of the sort just clear and simple don't matter who you are fuc...",0.000000,"[1.0, 0.0, 0.0]","[1.0, 0.0, 0.0]",0,0,0.001120,"[0.7210448, 0.046371896, 0.23258336]","[0.6872679, 0.04631285, 0.2664192]",0-0


Saved: context_embedding_outputs/context_embedding_true_vs_model_EL_EC.csv


## 17. Inspect High-Impact Context Examples

In [25]:
if "extreme_df" in globals() and len(extreme_df) > 0:
    top_row = extreme_df.sort_values("js", ascending=False).iloc[0]

    print("Comment:")
    print(top_row["text"])

    print("\nPrediction EL:")
    print(top_row["pred_extremely_liberal"])

    print("\nPrediction EC:")
    print(top_row["pred_extremely_conservative"])

    print("\nContext EL:")
    print(top_row["context_extremely_liberal"])

    print("\nContext EC:")
    print(top_row["context_extremely_conservative"])


Comment:
Send all these asses back !

Prediction EL:
[0.36457667 0.06890402 0.56651926]

Prediction EC:
[0.3074006  0.07150222 0.62109715]

Context EL:
### COMMENT TO CLASSIFY
Send all these asses back !

### ANNOTATOR IDEOLOGY
extremely_liberal

### RETRIEVED BACKGROUND
Free migration:
The concept of free migration has been debated globally, with various perspectives on its moral, economic, and cultural implications. Some argue that it's a human right, citing religious teachings such as Buddhism and Christianity, which emphasize hospitality and equality towards strangers. Others propose that open borders would lead to economic gains through increased labor mobility and reduced production efficiency gaps between countries. However, arguments against free migration often focus on economic, cultural, or security concerns, with some advocating for stricter immigration policies and border control.

Context EC:
### COMMENT TO CLASSIFY
Send all these asses back !

### ANNOTATOR IDEOLOGY
extr

## 18. Interpretation Guide

Compare this model against:

1. no-context subgroup embedding baseline,
2. strong FiLM no-context baseline,
3. later context + token model,
4. later context + strong FiLM model.

Important comparisons:

```text
1. Overall KL / JS
2. Macro F1
3. Class-2 recall
4. Whether class 1 is ever predicted
5. Mean counterfactual JS
6. EL↔EC model JS
7. True-vs-model capture ratio
```

A useful result is not only lower KL. If context increases counterfactual subgroup sensitivity without destroying overall metrics, that still supports the dissertation hypothesis.


In [ ]:
context_df["context_input_text"].nunique()

9087

In [ ]:
len(context_df)

9090

In [ ]:
context_df["text"].nunique()

3797

In [ ]:
for i in range(5):
    print("="*80)
    print(context_df.iloc[i]["context_input_text"])

Annotator ideology: extremely_liberal.

Retrieved background 1 (Left-wing politics):
Left-wing politics emphasizes social equality and egalitarianism, often opposing social hierarchies and advocating for the empowerment of marginalized groups. Leftist ideologies vary widely but typically involve a concern for those disadvantaged by society and a belief in reducing or abolishing unjustified inequalities through radical means. Key concepts include justice, solidarity, cultural pluralism, and freedom from forceful control or coercion. The left is associated with popular or state control of major institutions, trade unions, and critiques of capitalism, globalization, and environmental degradation, which disproportionately affect poorer communities.

Retrieved background 2 (Free migration):
The concept of free migration has been debated globally, with various perspectives on its moral, economic, and cultural implications. Some argue that it's a human right, citing religious teachings such a